# 04 - Time Series Analysis & Change Detection

**Arab Youth Space Hackathon 2026** | Team: ليوا الخضراء | Theme: Precision Agriculture & Crop Intelligence

This notebook covers:
- Multi-temporal Sentinel-2 data stacking
- Cloud masking and quality filtering
- Vegetation index time series construction
- Phenology analysis and crop calendar
- Change detection (NDVI differencing, BFAST concepts)
- Anomaly detection for crop stress

Target: Liwa Oasis growing season monitoring (Oct 2024 - Sep 2025)

In [ ]:
# Automatic dependency check & installation for Kaggle/Colab/Local environments
# The mechanistic justification for this step is: Dynamically inspecting and installing required packages (planetary-computer, pystac-client, rasterio, geopandas, folium, h5py) prevents runtime ModuleNotFoundError exceptions across Kaggle runtime environments.
import subprocess
import sys

def install_if_missing(package, pip_name=None):
    if pip_name is None:
        pip_name = package
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

install_if_missing("planetary_computer", "planetary-computer")
install_if_missing("pystac_client", "pystac-client")
install_if_missing("pystac", "pystac")
install_if_missing("rasterio", "rasterio")
install_if_missing("geopandas", "geopandas")
install_if_missing("folium", "folium")
install_if_missing("h5py", "h5py")

!pip -q install stackstac planetary-computer geopandas rasterio rioxarray xarray matplotlib scipy tqdm

In [ ]:
import numpy as np
import xarray as xr
import rioxarray
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from datetime import datetime
import planetary_computer as pc
from pystac_client import Client
import stackstac

# Liwa AOI
LIWA_BBOX = [53.5, 22.8, 54.2, 23.3]
GROWING_SEASON = "2024-10-01/2025-09-30"

## 1. Sentinel-2 Time Series Collection

In [ ]:
def search_sentinel2_timeseries(bbox, timerange, max_cloud=30, max_items=50):
    """Search for Sentinel-2 L2A scenes over time range."""
    catalog = Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace
    )
    
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=bbox,
        datetime=timerange,
        query={"eo:cloud_cover": {"lt": max_cloud}},
        max_items=max_items
    )
    
    items = list(search.items())
    print(f"Found {len(items)} scenes")
    
    # Sort by date
    items.sort(key=lambda x: x.datetime)
    
    for item in items:
        print(f"  {item.datetime.date()} - cloud: {item.properties.get('eo:cloud_cover')}% - {item.id}")
    
    return items

In [ ]:
def stack_sentinel2_timeseries(items, bbox, bands=None, resolution=10, epsg=32640):
    """Stack Sentinel-2 scenes into a time series DataArray."""
    if bands is None:
        bands = ["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12", "SCL"]
    
    stack = stackstac.stack(
        items,
        assets=bands,
        bounds=bbox,
        resolution=resolution,
        epsg=epsg,
        chunksize=2048,
        sortby_date=True
    )
    
    # Add time coordinate
    dates = [item.datetime for item in items]
    stack = stack.assign_coords(time=dates)
    
    print(f"Stack shape: {stack.shape}")  # (time, band, y, x)
    print(f"Time range: {stack.time.values[0]} to {stack.time.values[-1]}")
    print(f"Bands: {list(stack.band.values)}")
    
    return stack

## 2. Cloud Masking with SCL (Scene Classification Layer)

In [ ]:
def apply_scl_cloud_mask(stack, scl_band="SCL"):
    """
    Apply cloud mask using Sentinel-2 Scene Classification Layer.
    
    SCL classes to mask (set to NaN):
    0: NO_DATA
    1: SATURATED_OR_DEFECTIVE
    3: CLOUD_SHADOWS
    8: CLOUD_MEDIUM_PROBABILITY
    9: CLOUD_HIGH_PROBABILITY
    10: THIN_CIRRUS
    11: SNOW_ICE
    """
    scl = stack.sel(band=scl_band)
    
    # Create cloud mask
    cloud_classes = [0, 1, 3, 8, 9, 10, 11]
    cloud_mask = xr.zeros_like(scl, dtype=bool)
    for cls in cloud_classes:
        cloud_mask = cloud_mask | (scl == cls)
    
    # Apply mask to all bands except SCL
    data_bands = [b for b in stack.band.values if b != scl_band]
    masked_stack = stack.sel(band=data_bands).copy()
    
    for band in data_bands:
        masked_stack.loc[dict(band=band)] = masked_stack.sel(band=band).where(~cloud_mask)
    
    # Add cloud mask as coordinate
    masked_stack = masked_stack.assign_coords(cloud_mask=(('time', 'y', 'x'), cloud_mask.values))
    
    cloud_pct = cloud_mask.mean(dim=['y', 'x']) * 100
    print(f"Cloud coverage per scene: {cloud_pct.values}")
    
    return masked_stack

## 3. Monthly Composites (Mosaicking)

In [ ]:
def create_monthly_composites(stack, method="median"):
    """Create monthly composites from time series."""
    # Resample to monthly frequency
    if method == "median":
        monthly = stack.resample(time="MS").median(skipna=True)
    elif method == "max":
        monthly = stack.resample(time="MS").max(skipna=True)
    elif method == "min":
        monthly = stack.resample(time="MS").min(skipna=True)
    else:
        raise ValueError(f"Unknown method: {method}")
    
    print(f"Monthly composites: {monthly.sizes['time']} months")
    for t in monthly.time.values:
        print(f"  {pd.Timestamp(t).strftime('%Y-%m')}")
    
    return monthly

## 4. Vegetation Index Time Series

In [ ]:
def compute_index_timeseries(stack, index_name):
    """Compute a vegetation index for each time step."""
    eps = 1e-6
    
    if index_name == "NDVI":
        nir = stack.sel(band="B08")
        red = stack.sel(band="B04")
        idx = (nir - red) / (nir + red + eps)
    elif index_name == "EVI":
        nir = stack.sel(band="B08")
        red = stack.sel(band="B04")
        blue = stack.sel(band="B02")
        idx = 2.5 * (nir - red) / (nir + 6*red - 7.5*blue + 1 + eps)
    elif index_name == "SAVI":
        nir = stack.sel(band="B08")
        red = stack.sel(band="B04")
        L = 0.5
        idx = (nir - red) / (nir + red + L) * (1 + L)
    elif index_name == "NDWI":
        green = stack.sel(band="B03")
        nir = stack.sel(band="B08")
        idx = (green - nir) / (green + nir + eps)
    elif index_name == "NDWI_Gao":
        nir = stack.sel(band="B08")
        swir = stack.sel(band="B11")
        idx = (nir - swir) / (nir + swir + eps)
    elif index_name == "MSI":
        nir = stack.sel(band="B08")
        swir = stack.sel(band="B11")
        idx = swir / nir
    elif index_name == "NMDI":
        nir = stack.sel(band="B08")
        swir1 = stack.sel(band="B11")
        swir2 = stack.sel(band="B12")
        idx = (nir - (swir1 - swir2)) / (nir + (swir1 - swir2) + eps)
    else:
        raise ValueError(f"Unknown index: {index_name}")
    
    idx.name = index_name
    return idx

In [ ]:
def extract_roi_timeseries(index_stack, roi_geometry, reducer="mean"):
    """Extract time series for a region of interest."""
    import geopandas as gpd
    from shapely.geometry import mapping
    
    # Clip to ROI
    clipped = index_stack.rio.clip([mapping(roi_geometry)], crs=index_stack.rio.crs, drop=True)
    
    if reducer == "mean":
        ts = clipped.mean(dim=["x", "y"], skipna=True)
    elif reducer == "median":
        ts = clipped.median(dim=["x", "y"], skipna=True)
    elif reducer == "std":
        ts = clipped.std(dim=["x", "y"], skipna=True)
    else:
        raise ValueError(f"Unknown reducer: {reducer}")
    
    return ts.to_pandas()

## 5. Phenology Analysis

In [ ]:
def detect_phenology_metrics(ts, index_name="NDVI"):
    """
    Detect key phenological metrics from vegetation index time series.
    
    Returns:
    - SOS: Start of Season
    - POS: Peak of Season
    - EOS: End of Season
    - LOS: Length of Season
    - Peak value
    """
    from scipy.signal import find_peaks
    
    values = ts.values
    dates = ts.index
    
    # Smooth the time series
    from scipy.ndimage import gaussian_filter1d
    smoothed = gaussian_filter1d(values, sigma=1)
    
    # Find peaks
    peaks, properties = find_peaks(smoothed, distance=5, prominence=0.05)
    
    if len(peaks) == 0:
        return None
    
    # Main peak (highest)
    main_peak = peaks[np.argmax(smoothed[peaks])]
    pos_date = dates[main_peak]
    pos_value = smoothed[main_peak]
    
    # SOS: first date where value exceeds 20% of peak
    threshold = 0.2 * pos_value
    sos_candidates = np.where(smoothed[:main_peak] > threshold)[0]
    sos_date = dates[sos_candidates[0]] if len(sos_candidates) > 0 else dates[0]
    
    # EOS: last date where value exceeds 20% of peak
    eos_candidates = np.where(smoothed[main_peak:] > threshold)[0]
    eos_date = dates[main_peak + eos_candidates[-1]] if len(eos_candidates) > 0 else dates[-1]
    
    los_days = (eos_date - sos_date).days
    
    return {
        "SOS": sos_date,
        "POS": pos_date,
        "EOS": eos_date,
        "LOS_days": los_days,
        "Peak_value": pos_value,
        "Peak_date": pos_date
    }

## 6. Change Detection

In [ ]:
def compute_change_detection(stack, date_before, date_after, index_name="NDVI"):
    """Compute change between two dates using index differencing."""
    idx_before = compute_index_timeseries(stack.sel(time=date_before), index_name)
    idx_after = compute_index_timeseries(stack.sel(time=date_after), index_name)
    
    change = idx_after - idx_before
    change.name = f"{index_name}_change_{date_after}_{date_before}"
    
    return change


def threshold_change(change, threshold=-0.1):
    """Create binary change mask from continuous change."""
    # For NDVI: negative change = vegetation loss
    mask = change < threshold
    return mask.astype(np.uint8)

## 7. Anomaly Detection (Z-score based)

In [ ]:
def compute_anomalies(index_stack, window=5):
    """Compute z-score anomalies using rolling window."""
    # Rolling mean and std
    rolling_mean = index_stack.rolling(time=window, center=True, min_periods=2).mean()
    rolling_std = index_stack.rolling(time=window, center=True, min_periods=2).std()
    
    # Z-score
    anomalies = (index_stack - rolling_mean) / (rolling_std + 1e-6)
    anomalies.name = f"{index_stack.name}_anomaly"
    
    return anomalies

## 8. Visualization

In [ ]:
def plot_timeseries(ts_dict, title="Vegetation Index Time Series"):
    """Plot multiple time series."""
    plt.figure(figsize=(14, 6))
    for name, ts in ts_dict.items():
        plt.plot(ts.index, ts.values, marker='o', label=name, linewidth=2)
    
    plt.xlabel('Date')
    plt.ylabel('Index Value')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_index_map_timeseries(index_stack, dates, ncols=3):
    """Plot spatial maps for multiple dates."""
    n = len(dates)
    nrows = (n + ncols - 1) // ncols
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
    axes = axes.flatten() if n > 1 else [axes]
    
    for ax, date in zip(axes, dates):
        data = index_stack.sel(time=date)
        im = ax.imshow(data, cmap='RdYlGn', vmin=-0.2, vmax=1.0)
        ax.set_title(f"{pd.Timestamp(date).strftime('%Y-%m-%d')}")
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    
    for ax in axes[n:]:
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()